# Dataset IoT com InfluxDB

Este notebook demonstra operacoes basicas com o dataset `iot_telemetry_data.csv` usando o InfluxDB 2.x: criar bucket, converter registros para line protocol, inserir dados, consultar e deletar dados.

## Configuracao

Os valores abaixo sao os mesmos definidos no `docker-compose.yml`.

In [ ]:
import csv
import json
from decimal import Decimal
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / 'iot_telemetry_data.csv'

INFLUX_URL = 'http://localhost:8086'
INFLUX_ORG = 'ecom041'
INFLUX_BUCKET = 'iot_telemetry'
INFLUX_TOKEN = 'ecom041-token'
MEASUREMENT = 'environmental_telemetry'

CSV_PATH.exists(), CSV_PATH

## Funcoes auxiliares para a API HTTP do InfluxDB

In [ ]:
def http_request(method, path, body=None, content_type='application/json'):
    if isinstance(body, str):
        body = body.encode('utf-8')
    headers = {'Authorization': f'Token {INFLUX_TOKEN}'}
    if body is not None:
        headers['Content-Type'] = content_type
    request = Request(f'{INFLUX_URL}{path}', data=body, headers=headers, method=method)
    try:
        with urlopen(request, timeout=30) as response:
            return response.read()
    except HTTPError as exc:
        details = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Erro HTTP {exc.code}: {details}') from exc
    except URLError as exc:
        raise RuntimeError('Nao foi possivel conectar ao InfluxDB. Verifique se o Docker esta rodando.') from exc


def get_org_id():
    data = http_request('GET', f'/api/v2/orgs?org={quote(INFLUX_ORG)}')
    payload = json.loads(data.decode('utf-8'))
    return payload['orgs'][0]['id']


def ensure_bucket():
    data = http_request('GET', f'/api/v2/buckets?name={quote(INFLUX_BUCKET)}')
    payload = json.loads(data.decode('utf-8'))
    if payload.get('buckets'):
        print(f'Bucket ja existe: {INFLUX_BUCKET}')
        return
    body = {'orgID': get_org_id(), 'name': INFLUX_BUCKET, 'retentionRules': []}
    http_request('POST', '/api/v2/buckets', body=json.dumps(body))
    print(f'Bucket criado: {INFLUX_BUCKET}')

## Conversao do CSV para line protocol

In [ ]:
def escape_tag(value):
    return value.replace('\\', '\\\\').replace(' ', '\\ ').replace(',', '\\,').replace('=', '\\=')


def bool_to_lp(value):
    return 'true' if value.strip().lower() == 'true' else 'false'


def row_to_line_protocol(row):
    timestamp_ns = int(Decimal(row['ts']) * Decimal('1000000000'))
    device = escape_tag(row['device'])
    fields = [
        f"co={float(row['co'])}",
        f"humidity={float(row['humidity'])}",
        f"light={bool_to_lp(row['light'])}",
        f"lpg={float(row['lpg'])}",
        f"motion={bool_to_lp(row['motion'])}",
        f"smoke={float(row['smoke'])}",
        f"temp={float(row['temp'])}",
    ]
    return f"{MEASUREMENT},device={device} {','.join(fields)} {timestamp_ns}"


with CSV_PATH.open('r', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    first_row = next(reader)

first_row, row_to_line_protocol(first_row)

## Criar bucket

In [ ]:
ensure_bucket()

## Inserir dados

Para testes, comece com poucas linhas. Depois, altere `LIMIT = None` para inserir o CSV inteiro.

In [ ]:
LIMIT = 1000
BATCH_SIZE = 5000

write_path = f'/api/v2/write?org={quote(INFLUX_ORG)}&bucket={quote(INFLUX_BUCKET)}&precision=ns'
batch = []
total = 0

with CSV_PATH.open('r', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    for index, row in enumerate(reader):
        if LIMIT is not None and index >= LIMIT:
            break
        batch.append(row_to_line_protocol(row))
        if len(batch) >= BATCH_SIZE:
            http_request('POST', write_path, body='\n'.join(batch), content_type='text/plain')
            total += len(batch)
            batch.clear()

if batch:
    http_request('POST', write_path, body='\n'.join(batch), content_type='text/plain')
    total += len(batch)

total

## Consultar dados

In [ ]:
flux = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: 2020-07-12T00:00:00Z, stop: 2020-07-20T00:00:00Z)
  |> filter(fn: (r) => r._measurement == "{MEASUREMENT}")
  |> filter(fn: (r) => r._field == "temp" or r._field == "humidity")
  |> limit(n: 20)
'''

body = {'query': flux, 'type': 'flux'}
result = http_request('POST', f'/api/v2/query?org={quote(INFLUX_ORG)}', body=json.dumps(body))
print(result.decode('utf-8', errors='replace'))

## Deletar dados inseridos

In [ ]:
delete_body = {
    'start': '2020-07-12T00:00:00Z',
    'stop': '2020-07-20T00:00:00Z',
    'predicate': f'_measurement="{MEASUREMENT}"',
}
delete_path = f'/api/v2/delete?org={quote(INFLUX_ORG)}&bucket={quote(INFLUX_BUCKET)}'

# Descomente para executar a exclusao:
# http_request('POST', delete_path, body=json.dumps(delete_body))